# steps taken

- Load dependencies, 
- define self rag pipeline, load model + tokenizer 
- load datasets 
- Build corpus
- create llm as a judge for eval
- evaluate


## dependencies

In [41]:
!pip install -q \
    transformers \
    accelerate \
    faiss-cpu \
    sentence-transformers \
    datasets \
    torch \
    tqdm \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    scikit-learn \
    nltk \
    rouge-score \
    bert-score \
    evaluate \
    bitsandbytes \
    cohere

# !{sys.executable} -m pip install sentencepiece tiktoken
# !{sys.executable} -m pip install protobuf sentencepiece --upgrade

In [42]:
import os
import re
import json
import time
import warnings
import logging
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
import cohere
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import re
from dataclasses import dataclass
import nltk
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer
import evaluate

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

Device: cuda | PyTorch: 2.7.1+cu118


In [43]:
import os
os.environ["COHERE_API_KEY"] = "Swm9RbtxGtMwCe2ynd41NbriMjA4IE9wSY9IWmx0"

## self rag pipeline


In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────

@dataclass
class Config:
    model_name:      str   = "selfrag/selfrag_llama2_7b"
    max_new_tokens:  int   = 256
    temperature:     float = 1.0    # FIX 2: was 0.1 — too low suppresses special tokens
    do_sample:       bool  = False  # FIX 2: greedy decoding for reliable Self-RAG tokens
    top_p:           float = 1.0
    top_k:           int   = 3

    # Reflection tokens
    retrieve_token:            str = "[Retrieval]"
    no_retrieve_token:         str = "[No Retrieval]"
    relevant_token:            str = "[Relevant]"
    irrelevant_token:          str = "[Irrelevant]"
    partially_relevant_token:  str = "[Partially relevant]"
    fully_supported_token:     str = "[Fully supported]"
    partially_supported_token: str = "[Partially supported]"
    no_support_token:          str = "[No support / Contradictory]"

CFG = Config()

SELFRAG_SPECIAL_TOKENS = [
    "[Retrieval]", "[No Retrieval]",
    "[Relevant]", "[Irrelevant]", "[Partially relevant]",
    "[Fully supported]", "[Partially supported]", "[No support / Contradictory]",
    "[Utility:1]", "[Utility:2]", "[Utility:3]", "[Utility:4]", "[Utility:5]",
]

def register_selfrag_tokens(model, tokenizer):
    tokenizer.add_special_tokens({"additional_special_tokens": SELFRAG_SPECIAL_TOKENS})
    model.resize_token_embeddings(len(tokenizer))
    print(f"Special tokens registered — vocab size now {len(tokenizer)}")
    return model, tokenizer

class RetrievalCorpus:
    """TF-IDF retriever over a flat list of passage dicts."""

    def __init__(self, passages: List[Dict]):
        self.passages = passages
        texts = [p["passage"] for p in passages]
        self.vectorizer = TfidfVectorizer(stop_words="english", max_features=50_000)
        self.matrix     = self.vectorizer.fit_transform(texts)

    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        q_vec  = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.matrix).flatten()
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [
            {**self.passages[i], "score": float(scores[i])}
            for i in top_idx
            if scores[i] > 0
        ]

@dataclass
class SelfRAGOutput:
    question:            str
    retrieved_passages:  List[Dict]
    raw_generation:      str
    final_answer:        str
    reflection_tokens:   Dict[str, Any]
    retrieval_triggered: bool
    latency_ms:          float


class SelfRAGPipeline:
    """
    Implements the Self-RAG inference loop:
      1. Prompt the model; detect [Retrieval] token in output.
      2. If [Retrieval] is present, fetch top-k passages.
      3. Re-prompt with retrieved context; generate final answer.
      4. Parse all reflection tokens from the output.
    """

    REFLECTION_PATTERN = re.compile(
        r"(\[Retrieval\]|\[No Retrieval\]|\[Relevant\]|\[Irrelevant\]"
        r"|\[Partially relevant\]|\[Fully supported\]|\[Partially supported\]"
        r"|\[No support / Contradictory\]|\[Utility:\d\])"
    )

    def __init__(self, model, tokenizer, corpus: RetrievalCorpus, cfg: Config):
        self.model     = model
        self.tokenizer = tokenizer
        self.corpus    = corpus
        self.cfg       = cfg


    def _format_prompt_no_context(self, question: str) -> str:
        return (
            "### Instruction:\n"
            f"{question}\n\n"
            "### Response:\n"
        )

    def _format_prompt_with_context(self, question: str, passages: List[Dict]) -> str:
        ctx_lines = []
        for i, p in enumerate(passages, 1):
            ctx_lines.append(f"[{i}] {p['passage'].strip()}")
        context_block = "\n".join(ctx_lines)
        return (
            "### Instruction:\n"
            f"{question}\n\n"
            "### Context:\n"
            f"{context_block}\n\n"
            "### Response:\n"
        )


    @torch.inference_mode()
    def _generate(self, prompt: str) -> str:
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
        out = self.model.generate(
            **inputs,
            max_new_tokens=self.cfg.max_new_tokens,
            do_sample=self.cfg.do_sample,
            temperature=self.cfg.temperature if self.cfg.do_sample else None,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
    
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(new_tokens, skip_special_tokens=False)
    
        del inputs, out, new_tokens
        torch.cuda.empty_cache()
    
        return text


    def _parse_reflection_tokens(self, text: str) -> Dict[str, Any]:
        found = self.REFLECTION_PATTERN.findall(text)
        tokens = {
            "retrieval_decision": None,
            "relevance":          [],
            "groundedness":       None,
            "utility":            None,
            "all_tokens":         found,
        }
        for tok in found:
            if tok in (CFG.retrieve_token, CFG.no_retrieve_token):
                tokens["retrieval_decision"] = tok
            elif tok in (CFG.relevant_token, CFG.irrelevant_token, CFG.partially_relevant_token):
                tokens["relevance"].append(tok)
            elif tok in (CFG.fully_supported_token, CFG.partially_supported_token, CFG.no_support_token):
                tokens["groundedness"] = tok
            elif tok.startswith("[Utility:"):
                tokens["utility"] = int(tok.replace("[Utility:", "").replace("]", ""))
        return tokens


    def _clean_answer(self, text: str) -> str:
        """Strip reflection tokens and EOS artifacts from the final answer."""
        cleaned = self.REFLECTION_PATTERN.sub("", text)
        # Also strip EOS token string if it leaked through
        cleaned = cleaned.replace("</s>", "").replace("<s>", "")
        cleaned = re.sub(r"\s+", " ", cleaned).strip()
        return cleaned


    def predict(self, question: str) -> SelfRAGOutput:
        t0 = time.time()

        # Step 1: Initial generation without context
        prompt_initial = self._format_prompt_no_context(question)
        initial_output = self._generate(prompt_initial)

        # Step 2: Decide whether retrieval is needed
        retrieval_triggered = CFG.retrieve_token in initial_output
        retrieved_passages  = []
        raw_generation      = initial_output

        if retrieval_triggered:
            # Step 3: Retrieve and re-generate with context
            retrieved_passages = self.corpus.retrieve(question, top_k=self.cfg.top_k)
            if retrieved_passages:
                prompt_ctx     = self._format_prompt_with_context(question, retrieved_passages)
                raw_generation = self._generate(prompt_ctx)

        # Step 4: Parse reflection tokens and clean answer
        reflection = self._parse_reflection_tokens(raw_generation)
        final_ans  = self._clean_answer(raw_generation)

        latency_ms = (time.time() - t0) * 1000

        return SelfRAGOutput(
            question=question,
            retrieved_passages=retrieved_passages,
            raw_generation=raw_generation,
            final_answer=final_ans,
            reflection_tokens=reflection,
            retrieval_triggered=retrieval_triggered,
            latency_ms=latency_ms,
        )


In [45]:

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded — {n_params:.1f}M parameters")

JUDGE_DEVICE    = DEVICE
judge_model     = model
judge_tokenizer = tokenizer

print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.1f} GB")


Loading weights: 100%|██████████| 291/291 [00:14<00:00, 20.42it/s]


Model loaded — 6738.5M parameters
GPU memory allocated: 15.3 GB
GPU memory reserved:  16.0 GB


## dataset loading

In [ ]:


# ══════════════════════════════════════════════════════════════════
# SHARED CHUNKER
# ══════════════════════════════════════════════════════════════════

def chunk_text(text: str, size: int = 200, stride: int = 50) -> List[str]:
    words  = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunk = " ".join(words[start : start + size])
        if chunk.strip():
            chunks.append(chunk)
        if start + size >= len(words):
            break
        start += size - stride
    return chunks


# ══════════════════════════════════════════════════════════════════
# DATASET 1 — NBA Rules  (nba_rules_raw.json)
# ══════════════════════════════════════════════════════════════════

def load_nba_corpus(path: str = "nba_rules_raw.json") -> List[Dict]:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    passages = []
    for page in raw["pages"]:
        page_num = page["page"]
        text     = page["text"].strip()

        # Skip index / TOC pages and nearly-empty pages
        if page_num <= 8 or len(text.split()) < 50:
            continue

        for chunk in chunk_text(text, size=200, stride=50):
            passages.append({
                "passage": chunk,
                "source":  f"NBA Rules p.{page_num}",
                "dataset": "nba_rules",
            })

    print(f"NBA corpus    → {len(passages):,} passages  ({len(raw['pages'])} pages)")
    return passages


# ══════════════════════════════════════════════════════════════════
# DATASET 2 — Wiki Corpus  (wiki_corpus.json)
# ══════════════════════════════════════════════════════════════════

def load_wiki_corpus(path: str = "wiki_corpus.json") -> List[Dict]:
    with open(path, encoding="utf-8") as f:
        articles = json.load(f)

    passages = []
    for art in articles:
        text = art.get("clean_content") or art.get("content", "")
        if not text.strip():
            continue
        for chunk in chunk_text(text, size=200, stride=50):
            passages.append({
                "passage": chunk,
                "source":  art["title"],
                "dataset": "wiki_corpus",
            })

    print(f" Wiki corpus   → {len(passages):,} passages  ({len(articles)} articles)")
    return passages


# ══════════════════════════════════════════════════════════════════
# DATASET 3 — Sci Wiki Unprocessed  (sci_wiki_unprocessed.jsonl)
# ══════════════════════════════════════════════════════════════════

def load_sci_wiki_corpus(path: str = "sci_wiki_unprocessed.jsonl") -> List[Dict]:
    records = [
        json.loads(line)
        for line in Path(path).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    passages = []
    for rec in records:
        for chunk in chunk_text(rec["text"], size=200, stride=50):
            passages.append({
                "passage": chunk,
                "source":  rec["title"],
                "url":     rec.get("url", ""),
                "dataset": "sci_wiki",
            })

    print(f"Sci-wiki corpus → {len(passages):,} passages  ({len(records)} articles)")
    return passages


# ══════════════════════════════════════════════════════════════════
# EVAL DATASET  (generic loader for all eval JSONs)
# ══════════════════════════════════════════════════════════════════

def load_eval_dataset(path: str, dataset_name: str) -> List[Dict]:
    """
    Loads evaluation dataset in format:
    [
      {
        "question": "...",
        "reference_answer": "..."
      }
    ]

    Returns standardized structure for RAG evaluation.
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    eval_samples = []
    for i, item in enumerate(data):
        question = item.get("question", "").strip()
        answer   = item.get("reference_answer", "").strip()

        if not question:
            continue

        eval_samples.append({
            "id": f"{dataset_name}_{i}",
            "question": question,
            "reference_answer": answer,
            "dataset": dataset_name,
        })

    print(f"{dataset_name} eval → {len(eval_samples):,} questions")
    return eval_samples


# ══════════════════════════════════════════════════════════════════
# LOAD EVERYTHING
# ══════════════════════════════════════════════════════════════════

nba_corpus_passages  = load_nba_corpus("nba_rules_raw.json")
wiki_corpus_passages = load_wiki_corpus("wiki_corpus.json")
sci_wiki_passages    = load_sci_wiki_corpus("sci_wiki_unprocessed.jsonl")
nba_eval         = load_eval_dataset("../test_questions_nba.json", "nba_rules")
wiki_corpus_eval = load_eval_dataset("../test_questions_wiki.json", "wiki_corpus")
sci_wiki_eval    = load_eval_dataset("../test_questions_sci.json", "sci_wiki")

# ── Deduplicate across all corpora ────────────────────────────────
all_passages = nba_corpus_passages + wiki_corpus_passages + sci_wiki_passages
seen_texts, unique_passages = set(), []
for p in all_passages:
    if p["passage"] not in seen_texts:
        seen_texts.add(p["passage"])
        unique_passages.append(p)

print(f"\nTotal unique passages for corpus : {len(unique_passages):,}")
print(f" NBA eval questions               : {len(nba_eval)}")


corpus = RetrievalCorpus(unique_passages)

## final eval

In [52]:
import transformers
transformers.logging.set_verbosity_error()

def extract_context_from_output(output) -> str:
    if not getattr(output, "retrieved_passages", None):
        return ""

    lines = []
    for i, p in enumerate(output.retrieved_passages, 1):
        if isinstance(p, dict):
            text   = p.get("passage", p.get("text", "")).strip()
            source = p.get("source", p.get("title", ""))
        else:
            text   = getattr(p, "passage", getattr(p, "text", "")).strip()
            source = getattr(p, "source", getattr(p, "title", ""))

        lines.append(f"[{i}] ({source}) {text}")

    return "\n".join(lines)



REFLECTION_STRIP = re.compile(
    r"(\[Retrieve\]|\[No Retrieve\]|\[Relevant\]|\[Irrelevant\]"
    r"|\[Partially relevant\]|\[Fully supported\]|\[Partially supported\]"
    r"|\[No support / Contradictory\]|\[Utility:\d\]|</s>|<s>)"
)

def build_corpus_passages(datasets: dict) -> list:
    seen, unique = set(), []
    for dataset_name, passages in datasets.items():
        for p in passages:
            text = p.get("passage", "").strip()
            if text and text not in seen:
                seen.add(text)
                unique.append(p)
    return unique


co = cohere.Client(os.environ.get("COHERE_API_KEY", "YOUR_COHERE_API_KEY"))

def chat_with_retry(max_retries=4, **kwargs):
    for attempt in range(max_retries):
        try:
            return co.chat(**kwargs)
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 15 * (2 ** attempt)
                print(f"  Rate limited — waiting {wait}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Max retries exceeded on Cohere rate limit.")


def llm_judge_local(question: str, context: str, answer_text: str,
                    reference_answer: str = "") -> dict:
    prompt = f"""You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems.
        Score the answer below on three criteria using a 1–5 integer scale:
          1 = very poor  |  3 = acceptable  |  5 = excellent
        
        Criteria:
        - faithfulness:      Every claim in the answer is directly supported by the provided context.
                             Penalise heavily for hallucination or any statement not grounded in the context.
                             If context is empty, score 1.
        - answer_relevance:  The answer directly and completely addresses the question with a real answer.
                             IMPORTANT: If the answer says it "cannot answer", "no information", "not sure",
                             or anything similar — that is a FAILED answer. Score it 1.
                             Compare against the reference answer if provided.
        - context_relevance: The retrieved context actually contained information useful for answering
                             the question. If context is empty or off-topic, score 1.
        
        QUESTION:
        {question}
        
        REFERENCE ANSWER:
        {reference_answer if reference_answer else "(none provided)"}
        
        RETRIEVED CONTEXT:
        {context if context else "(no context retrieved)"}
        
        ANSWER:
        {answer_text}
        
        Return ONLY a valid JSON object (no extra text):
        {{
          "faithfulness": <1-5>,
          "answer_relevance": <1-5>,
          "context_relevance": <1-5>,
          "reasoning": "<one sentence explaining the scores>"
        }}
        """
    response = chat_with_retry(
        model       = "command-r7b-12-2024",
        message     = prompt,
        temperature = 0,
    )
    raw = response.text.strip()

    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()

    m = re.search(r"\{.*?\}", raw, re.DOTALL)
    if m:
        raw = m.group(0)

    try:
        scores = json.loads(raw)
        return {
            "faithfulness":      max(1, min(5, int(scores.get("faithfulness", 1)))),
            "answer_relevance":  max(1, min(5, int(scores.get("answer_relevance", 1)))),
            "context_relevance": max(1, min(5, int(scores.get("context_relevance", 1)))),
            "reasoning":         scores.get("reasoning", ""),
        }
    except (json.JSONDecodeError, ValueError):
        def extract_score(key):
            m = re.search(rf'"{key}"\s*:\s*([1-5])', raw)
            return int(m.group(1)) if m else 1
        print(f"  ⚠️  parse failed: {repr(raw[:200])}")
        return {
            "faithfulness":      extract_score("faithfulness"),
            "answer_relevance":  extract_score("answer_relevance"),
            "context_relevance": extract_score("context_relevance"),
            "reasoning":         "parse error",
        }


In [53]:
pipeline_selfrag = SelfRAGPipeline(model, tokenizer, corpus, CFG)
print("pipeline_selfrag ready")

pipeline_selfrag ready


In [58]:
raw_datasets = {
    "nba":      nba_corpus_passages,    
    "wiki":     wiki_corpus_passages,   
    "sci_wiki": sci_wiki_passages,      
}

eval_datasets = {
    "nba":      nba_eval,
    "wiki":     wiki_corpus_eval,       
    "sci_wiki": sci_wiki_eval,   
}

all_summaries = []

# Build all three corpora
prebuilt_corpora = {
    name: RetrievalCorpus(build_corpus_passages(raw_datasets))
    for name in eval_datasets
}

for DATASET_NAME, test_questions in eval_datasets.items():
    if not test_questions:
        continue

    
    print(f"\n{'='*55}")
    print(f"  Evaluating: {DATASET_NAME}  ({len(test_questions)} questions)")
    print(f"{'='*55}")
    
    pipeline_selfrag.corpus = prebuilt_corpora[DATASET_NAME] 

    # Resume previous results
    eval_results = []
    done_questions = set()
    if os.path.exists(f"eval_results_selfrag_{DATASET_NAME}.json"):
        with open(f"eval_results_selfrag_{DATASET_NAME}.json", "r") as f:
            eval_results = json.load(f)
        done_questions = {r["question"] for r in eval_results}

    for item in test_questions:
        question = item["question"]
        if question in done_questions:
            continue

        try:
            output = pipeline_selfrag.predict(question)  
            ans = output.final_answer.strip()
            ctx = extract_context_from_output(output)

            scores = llm_judge_local(question, ctx, ans, reference_answer = item.get("reference_answer", ""))

        except Exception as e:
            print(f"Error for '{question}': {e}")
            continue

        eval_results.append({
            "question": question,
            "reference_answer": item.get("reference_answer", ""),
            "selfrag_answer": ans,
            "context": ctx,
            "retrieval_triggered": output.retrieval_triggered,
            "reflection_tokens": output.reflection_tokens["all_tokens"],
            **scores
        })

        EVAL_RESULTS_PATH = f"eval_results_selfrag_{DATASET_NAME}.json"
        # Save after each question
        with open(EVAL_RESULTS_PATH, "w") as f:
            json.dump(eval_results, f, indent=2, ensure_ascii=False)

        del output, scores
        torch.cuda.empty_cache()

    if not eval_results:
        print(f"  No results for {DATASET_NAME}")
        continue

    metrics  = ("faithfulness", "answer_relevance", "context_relevance")
    averages = {m: sum(r[m] for r in eval_results) / len(eval_results) for m in metrics}
    retrieval_rate = sum(r["retrieval_triggered"] for r in eval_results) / len(eval_results)

    print(f"\n  {'─'*40}")
    print(f"  Faithfulness         : {averages['faithfulness']:.2f} / 5")
    print(f"  Answer Relevance     : {averages['answer_relevance']:.2f} / 5")
    print(f"  Context Relevance    : {averages['context_relevance']:.2f} / 5")
    print(f"  Overall              : {sum(averages.values())/3:.2f} / 5")
    print(f"  Retrieval triggered  : {retrieval_rate:.1%}")
    print(f"  Saved to             : {EVAL_RESULTS_PATH}")

    all_summaries.append({
        "dataset":           DATASET_NAME,
        "n_questions":       len(eval_results),
        "faithfulness":      averages["faithfulness"],
        "answer_relevance":  averages["answer_relevance"],
        "context_relevance": averages["context_relevance"],
        "overall":           sum(averages.values()) / 3,
        "retrieval_rate":    retrieval_rate,
    })



  Evaluating: nba  (30 questions)

  ────────────────────────────────────────
  Faithfulness         : 3.73 / 5
  Answer Relevance     : 4.07 / 5
  Context Relevance    : 2.53 / 5
  Overall              : 3.44 / 5
  Retrieval triggered  : 46.7%
  Saved to             : eval_results_selfrag_sci_wiki.json

  Evaluating: wiki  (24 questions)

  ────────────────────────────────────────
  Faithfulness         : 3.92 / 5
  Answer Relevance     : 4.50 / 5
  Context Relevance    : 3.83 / 5
  Overall              : 4.08 / 5
  Retrieval triggered  : 87.5%
  Saved to             : eval_results_selfrag_sci_wiki.json

  Evaluating: sci_wiki  (24 questions)

  ────────────────────────────────────────
  Faithfulness         : 3.75 / 5
  Answer Relevance     : 4.67 / 5
  Context Relevance    : 3.75 / 5
  Overall              : 4.06 / 5
  Retrieval triggered  : 83.3%
  Saved to             : eval_results_selfrag_sci_wiki.json


In [59]:
print(f"\n\n{'='*65}")
print("  FINAL SUMMARY ACROSS ALL DATASETS")
print(f"{'='*65}")
summary_df = pd.DataFrame(all_summaries).set_index("dataset")
print(summary_df.round(3))



  FINAL SUMMARY ACROSS ALL DATASETS
          n_questions  faithfulness  answer_relevance  context_relevance  \
dataset                                                                    
nba                30         3.733             4.067              2.533   
wiki               24         3.917             4.500              3.833   
sci_wiki           24         3.750             4.667              3.750   

          overall  retrieval_rate  
dataset                            
nba         3.444           0.467  
wiki        4.083           0.875  
sci_wiki    4.056           0.833  
